## Step 0 — Install Packages & LLaMA-Factory

In [1]:
# 0-A: Fix numpy ABI first
!pip install --force-reinstall --no-deps numpy==2.0.2

# 0-B: Core training packages
!pip install --upgrade \
    scipy==1.14.1 \
    scikit-learn==1.5.2 \
    pandas==2.2.2 \
    transformers==4.51.3 \
    accelerate==1.2.1 \
    peft==0.14.0 \
    trl==0.13.0 \
    datasets==3.2.0 \
    bitsandbytes \
    json-repair==0.29.1 \
    rapidfuzz \
    wandb

# 0-C: LLaMA-Factory
!rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e .

# 0-D: Sanity check
import torch, transformers, numpy as np
print('✅ Torch        :', torch.__version__)
print('✅ Transformers :', transformers.__version__)
print('✅ NumPy        :', np.__version__)
print('✅ GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('   GPU name    :', torch.cuda.get_device_name(0))
    print('   VRAM        :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')


  Using cached numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
Using cached numpy-2.0.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (19.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
  Using cached accelerate-1.2.1-py3-none-any.whl.metadata (19 kB)
  Using cached peft-0.14.0-py3-none-any.whl.metadata (13 kB)
  Using cached trl-0.13.0-py3-none-any.whl.metadata (11 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached transformers-4.51.3-py3-none-any.whl (10.4 MB)
Using cached accelerate-1.2.1-py3-none-any.whl (336 kB)
Using cached peft-0.14.0-py3-none-any.whl (374 kB)
Using cached trl-0.13.0-py3-none-any.whl (293 kB)
Us

## Step 1 — Auth (WandB + HuggingFace)

> Requires Kaggle secrets: `WANDB_KEY` and `HF_TOKEN`.

In [2]:
from kaggle_secrets import UserSecretsClient
import wandb, os

user_secrets = UserSecretsClient()

wandb.login(key=user_secrets.get_secret('WANDB_KEY'))
print('✅ WandB authenticated')

hf_token = user_secrets.get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token
!huggingface-cli login --token $hf_token
print('✅ HuggingFace authenticated')


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mohmadazelmad3 (mohmadazelmad3-awalgpt) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ WandB authenticated

Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help

✅ HuggingFace authenticated


## Step 2 — Imports & Paths

In [3]:
import json, os, random
from os.path import join
from tqdm.auto import tqdm
from typing import List, Optional, Dict

# Paths
DATA_INPUT  = '/kaggle/input/datasets/thebasenews/awal-chatbotttt'
DATA_RAW    = join(DATA_INPUT, 'qa.json')
WORK_DIR    = '/kaggle/working/awal-chatbott'
SFT_DIR     = join(WORK_DIR, 'datasets', 'llamafactory-finetune-data')
OUTPUT_DIR  = join(WORK_DIR, 'models', 'awal-chatbot-lora')
MERGED_DIR  = '/kaggle/working/awal-merged-7b'

# HuggingFace Hub
HF_USERNAME = 'mohamedazelmad'
HF_REPO     = 'awal-gpt-v0.2-7b'
HF_REPO_ID  = f'{HF_USERNAME}/{HF_REPO}'

print(f'✅ SFT    : {SFT_DIR}')
print(f'✅ Output : {OUTPUT_DIR}')


✅ SFT    : /kaggle/working/awal-chatbott/datasets/llamafactory-finetune-data
✅ Output : /kaggle/working/awal-chatbott/models/awal-chatbot-lora


## Step 3 — System Prompt & Chat Builder

Fields aligned with `pipeline.py`: `intent`, `instruction`, `rag`, `entities`, `history`.

In [4]:
AWAL_SYSTEM_PROMPT = "\n".join([
    "Tu es AWAL — assistant IA specialise en Tamazight Amazigh marocaine (Tamazight Central).",
    "Tu reponds TOUJOURS en Tamazight Latin + Arabe.",
    "Forme sur IRCAM, TAWNZA et textes berberes authentiques.",
    "",
    "# CONTEXTE PIPELINE:",
    "- INTENTION: {intent}",
    "- INSTRUCTION: {instruction}",
    "- RAG_DICTIONARY: {rag}",
    "- ENTITIES_NER: {entities}",
    "- HISTORY: {history}",
    "",
    "# REGLES STRICTES:",
    "1. Format UNIQUEMENT:",
    "   latin: [reponse Tamazight Latin]",
    "   ar: [reponse Arabe]",
    "2. Genere l arabe d abord, traduis en Tamazight Latin.",
    "3. Les deux lignes disent EXACTEMENT la meme chose.",
    "4. N invente jamais — utilise le RAG si disponible.",
    "5. Aucun texte hors format.",
    "",
    "# EXEMPLES:",
    "User: Azul, amek tellid?",
    "latin: Azul! Labas, tanmirt. Mani tella netta?",
    "ar: \u0645\u0631\u062d\u0628\u0627\u064b\u061f \u0628\u062e\u064a\u0631\u060c \u0634\u0643\u0631\u0627\u064b. \u0648\u0643\u064a\u0641 \u062d\u0627\u0644\u0643 \u0623\u0646\u062a\u061f",
    "",
    "User: Traduis pain s tmazight",
    "latin: aghrum",
    "ar: \u0627\u0644\u062e\u0628\u0632",
])


def build_chat_messages(
    user_input: str,
    expected_output: str,
    pipeline_ctx: Optional[Dict] = None,
    history: Optional[List[Dict]] = None,
) -> List[Dict]:
    """Returns ChatML message list for LLaMA-Factory SFT."""
    ctx    = pipeline_ctx or {}
    system = AWAL_SYSTEM_PROMPT.format(
        intent      = ctx.get("intent",      "general"),
        instruction = ctx.get("instruction", "Reponds de facon utile et naturelle."),
        rag         = ctx.get("rag_context", "Aucun"),
        entities    = ctx.get("entities",    "Aucune"),
        history     = str(ctx.get("history", [])[-4:]),
    )
    msgs = [{"role": "system", "content": system}]
    for turn in (history or [])[-4:]:
        if "user"      in turn: msgs.append({"role": "user",      "content": turn["user"]})
        if "assistant" in turn: msgs.append({"role": "assistant", "content": turn["assistant"]})
    msgs.append({"role": "user",      "content": user_input})
    msgs.append({"role": "assistant", "content": expected_output})
    return msgs


print("✅ AWAL_SYSTEM_PROMPT ready (5 pipeline fields)")
print("✅ build_chat_messages ready")


✅ AWAL_SYSTEM_PROMPT ready (5 pipeline fields)
✅ build_chat_messages ready


## Step 4 — Load Raw Dataset

In [5]:
with open(DATA_RAW, encoding='utf-8') as f:
    raw_data = json.load(f)

random.Random(101).shuffle(raw_data)
print(f'✅ Loaded {len(raw_data):,} examples')
print(f'   Keys   : {list(raw_data[0].keys())}')
print(f'   Sample : {str(raw_data[0].get("input",""))[:80]}')


✅ Loaded 6,887 examples
   Keys   : ['input', 'output', 'intent']
   Sample : mata tga tmazight n Tinghir i tfrara n tgemmi


## Step 5 — Format → SFT (ChatML)

In [6]:
sft_data = []
skipped  = 0

for rec in tqdm(raw_data, desc='Formatting'):
    user_input      = rec.get('input',  '').strip()
    expected_output = rec.get('output', '').strip()
    if not user_input or not expected_output:
        skipped += 1
        continue

    ctx = {
        'intent':      rec.get('intent',      'general'),
        'instruction': rec.get('instruction', 'Reponds de facon utile et naturelle.'),
        'rag_context': rec.get('rag_context', 'Aucun'),
        'entities':    rec.get('entities',    'Aucune'),
        'history':     rec.get('history',     []),
    }
    msgs = build_chat_messages(user_input, expected_output, ctx, ctx['history'])
    sft_data.append({
        'system':      msgs[0]['content'],
        'instruction': user_input,
        'input':       '',
        'output':      expected_output,
        'history':     ctx['history'],
    })

random.Random(101).shuffle(sft_data)
print(f'\n✅ Formatted : {len(sft_data):,} examples')
print(f'   Skipped   : {skipped} (empty fields)')


Formatting:   0%|          | 0/6887 [00:00<?, ?it/s]


✅ Formatted : 6,887 examples
   Skipped   : 0 (empty fields)


## Step 6 — Split & Save  (90 / 10)

In [7]:
os.makedirs(SFT_DIR, exist_ok=True)

cut      = int(len(sft_data) * 0.90)
train_ds = sft_data[:cut]
val_ds   = sft_data[cut:]

with open(join(SFT_DIR, 'train.json'), 'w', encoding='utf-8') as f:
    json.dump(train_ds, f, ensure_ascii=False, default=str)

with open(join(SFT_DIR, 'val.json'), 'w', encoding='utf-8') as f:
    json.dump(val_ds, f, ensure_ascii=False, default=str)

print(f'✅ Train : {len(train_ds):,}  → {SFT_DIR}/train.json')
print(f'✅ Val   : {len(val_ds):,}   → {SFT_DIR}/val.json')


✅ Train : 6,198  → /kaggle/working/awal-chatbott/datasets/llamafactory-finetune-data/train.json
✅ Val   : 689   → /kaggle/working/awal-chatbott/datasets/llamafactory-finetune-data/val.json


## Step 7 — Register Datasets in LLaMA-Factory

In [8]:
info_path = './LLaMA-Factory/data/dataset_info.json'
if not os.path.exists(info_path):
    raise FileNotFoundError('LLaMA-Factory not found — run Step 0 first')

with open(info_path, 'r', encoding='utf-8') as f:
    info = json.load(f)

for name, fname in [('awal_chat_train', 'train.json'), ('awal_chat_val', 'val.json')]:
    info[name] = {
        'file_name': join(SFT_DIR, fname),
        'columns': {
            'prompt':   'instruction',
            'query':    'input',
            'response': 'output',
            'system':   'system',
            'history':  'history',
        },
    }

with open(info_path, 'w', encoding='utf-8') as f:
    json.dump(info, f, ensure_ascii=False, indent=2)

print('✅ awal_chat_train registered')
print('✅ awal_chat_val   registered')


✅ awal_chat_train registered
✅ awal_chat_val   registered


In [9]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ['WANDB_PROJECT']   = 'awal-chatbot'
os.environ['WANDB_LOG_MODEL'] = 'checkpoint'
os.environ['WANDB_MODE']      = 'offline'    # ← ADD
os.environ['WANDB_SILENT']    = 'true'       # ← ADD

ckpts    = [d for d in os.listdir(OUTPUT_DIR) if d.startswith('checkpoint-')] if os.path.isdir(OUTPUT_DIR) else []
has_ckpt = bool(ckpts)
if has_ckpt:
    last = max(ckpts, key=lambda x: int(x.split('-')[-1]))
    resume_line = 'resume_from_checkpoint: true'
else:
    resume_line = '# resume_from_checkpoint: false'

YAML = f'''### model
model_name_or_path: Qwen/Qwen2.5-3B-Instruct
trust_remote_code: true
quantization_bit: 4
quantization_method: bitsandbytes

upcast_layernorm: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 64
lora_alpha: 64
lora_dropout: 0.05
lora_target: q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj

### dataset
dataset: awal_chat_train
eval_dataset: awal_chat_val
template: qwen
cutoff_len: 1024
overwrite_cache: true
preprocessing_num_workers: 4

### output
output_dir: {OUTPUT_DIR}
logging_steps: 50
save_steps: 200
save_total_limit: 3
plot_loss: true
overwrite_output_dir: true
{resume_line}

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 16
learning_rate: 8e-5
num_train_epochs: 5
lr_scheduler_type: cosine
warmup_steps: 100
weight_decay: 0.01
fp16: true
ddp_timeout: 180000000


gradient_checkpointing: true

### eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 200

### wandb
report_to: wandb
run_name: awal-chatbot-7b-v3

push_to_hub: false
'''

YAML_PATH = '/kaggle/working/LLaMA-Factory/examples/train_lora/awal_chatbot_7b.yaml'
os.makedirs(os.path.dirname(YAML_PATH), exist_ok=True)
with open(YAML_PATH, 'w') as f:
    f.write(YAML)
print("oki")

oki


In [ ]:
import subprocess, os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

subprocess.run(
    f"cd /kaggle/working/LLaMA-Factory && "
    f"PYTORCH_ALLOC_CONF=expandable_segments:True "
    f"CUDA_VISIBLE_DEVICES=0,1 llamafactory-cli train {YAML_PATH}",
    shell=True, check=True
)

[INFO|2026-06-04 12:19:31] llamafactory.launcher:144 >> Initializing 2 distributed tasks at: 127.0.0.1:59291


W0604 12:19:33.650000 1488 torch/distributed/run.py:852] 
W0604 12:19:33.650000 1488 torch/distributed/run.py:852] *****************************************
W0604 12:19:33.650000 1488 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0604 12:19:33.650000 1488 torch/distributed/run.py:852] *****************************************


[INFO] Running in WANDB offline mode
[INFO] Running in WANDB offline mode


[W604 12:19:52.241707022 ProcessGroupNCCL.cpp:929] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())
[W604 12:19:52.337114824 ProcessGroupNCCL.cpp:929] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[INFO|2026-06-04 12:19:52] llamafactory.hparams.parser:144 >> Set `ddp_find_unused_parameters` to False in DDP training since LoRA is enabled.
[INFO|2026-06-04 12:19:52] llamafactory.hparams.parser:523 >> Process rank: 0, world size: 2, device: cuda:0, distributed training: True, compute dtype: torch.float16


[INFO|configuration_utils.py:771] 2026-06-04 12:19:52,740 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-3B-Instruct/snapshots/aa8e72537993ba99e69dfaafa59ed015b17504d1/config.json
[INFO|configuration_utils.py:847] 2026-06-04 12:19:52,746 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 2048,
  "initializer_range": 0.02,
  "intermediate_size": 11008,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",

... (24 outputs left out)

{'loss': '0.5087', 'grad_norm': '1.436', 'learning_rate': '5.222e-05', 'epoch': '2.32'}


 52%|█████▏    | 500/970 [1:20:38<2:03:55, 15.82s/it]

{'loss': '0.5219', 'grad_norm': '1.269', 'learning_rate': '4.519e-05', 'epoch': '2.578'}


 53%|█████▎    | 518/970 [1:25:23<1:59:21, 15.84s/it]

## Step 10 — Merge LoRA → Full Model

In [ ]:
import subprocess
ckpts = [d for d in os.listdir(OUTPUT_DIR) if d.startswith('checkpoint-')]
if not ckpts:
    raise RuntimeError('No checkpoint found')

BEST_CKPT    = max(ckpts, key=lambda x: int(x.split('-')[-1]))
ADAPTER_PATH = join(OUTPUT_DIR, BEST_CKPT)
os.makedirs(MERGED_DIR, exist_ok=True)

subprocess.run(
    f"cd /kaggle/working/LLaMA-Factory && llamafactory-cli export"
    f" --model_name_or_path {BASE_MODEL}"
    f" --adapter_name_or_path {ADAPTER_PATH}"
    f" --template qwen"
    f" --finetuning_type lora"
    f" --export_dir {MERGED_DIR}"
    f" --export_size 2"
    f" --export_device cpu"
    f" --export_legacy_format false",
    shell=True, check=True
)

## Step 11 — ✅ Test Merged Model

Runs 5 test cases covering greeting, translation, knowledge, culture, Arabizi input.  
Each response is checked for the required `latin: ... / ar: ...` format.

In [ ]:
import re, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print('Loading merged model...')
tok   = AutoTokenizer.from_pretrained(MERGED_DIR, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
print('✅ Model loaded\n')

TEST_CASES = [
    ('Azul, amek tellid?',                       'salutation'),
    ('Traduis pain s tmazight',                  'traduction'),
    ('Manta tga College of London?',             'question_info'),
    ('Amek yella Yennayer?',                     'culture'),
    ('righ adisingh maydyan l3asima n lmghrib',  'question_info'),
]

RE_LATIN = re.compile(r'latin\s*:\s*(.+?)(?=\nar\s*:|$)', re.DOTALL | re.IGNORECASE)
RE_AR    = re.compile(r'\bar\s*:\s*(.+?)$',               re.DOTALL | re.IGNORECASE)

def run_test(user_msg, intent='general'):
    system = AWAL_SYSTEM_PROMPT.format(
        intent='general', instruction='Reponds naturellement.',
        rag='Aucun', entities='Aucune', history='[]',
    )
    chat = [{'role':'system','content':system}, {'role':'user','content':user_msg}]
    text = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    ids  = tok(text, return_tensors='pt').input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=200, temperature=0.3,
            top_p=0.9, repetition_penalty=1.15, do_sample=True,
            pad_token_id=tok.eos_token_id,
        )
    resp = tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True).strip()
    ok   = bool(RE_LATIN.search(resp)) and bool(RE_AR.search(resp))
    return resp, '✅ PASS' if ok else '❌ FAIL (format error)'

print('=' * 60)
results = []
for i, (msg, intent) in enumerate(TEST_CASES, 1):
    print(f'\nTest {i}/{len(TEST_CASES)}  [{intent}]')
    print(f'  Input  : {msg}')
    resp, status = run_test(msg, intent)
    print(f'  Output : {resp[:200]}')
    print(f'  Status : {status}')
    results.append(status)

passed = sum(1 for r in results if 'PASS' in r)
print('\n' + '=' * 60)
print(f'RESULT: {passed}/{len(TEST_CASES)} passed')
if passed == len(TEST_CASES):
    print('All tests passed — ready to push!')
else:
    print('Some tests failed — check output format above before pushing.')


## Step 12 — Push to HuggingFace Hub

> Run only after Step 11 passes all tests.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, token=os.environ['HF_TOKEN'], exist_ok=True)

api.upload_folder(
    folder_path    = MERGED_DIR,
    repo_id        = HF_REPO_ID,
    token          = os.environ['HF_TOKEN'],
    commit_message = f'AWAL 7B SFT — {BEST_CKPT}',
)
print(f'Pushed to https://huggingface.co/{HF_REPO_ID}')
